In [ ]:
import pandas as pd
import numpy as np

WORKING_DAYS = 24
MACHINE_CAPACITY = 22

# =========================
# LOAD BOTH SHEETS
# =========================

hz = pd.read_excel("input.xlsx", sheet_name="HZ")
vt = pd.read_excel("input.xlsx", sheet_name="VT")

# =========================
# FUNCTION TO MELT MACHINES
# =========================

def expand_machines(df):

    machine_cols = [col for col in df.columns if col.startswith("Machine")]

    df_long = df.melt(
        id_vars=["Part","Inventory","Monthly_Indent","Category","Cycle time","Cavity"],
        value_vars=machine_cols,
        var_name="Machine_Col",
        value_name="Machine"
    )

    df_long = df_long.dropna(subset=["Machine"])

    return df_long

hz_long = expand_machines(hz)
vt_long = expand_machines(vt)

df = pd.concat([hz_long, vt_long], ignore_index=True)

# =========================
# CALCULATE RATE
# =========================

df["Rate"] = (3600 / df["Cycle time"]) * df["Cavity"]

# =========================
# MRP LOGIC
# =========================

df["Daily_Demand"] = df["Monthly_Indent"] / WORKING_DAYS
df["Safety"] = df["Daily_Demand"] * 3
df["Gap"] = df["Safety"] - df["Inventory"]
df["Trigger"] = np.where(df["Gap"] > 0, 1, 0)

# =========================
# CATEGORY TARGET
# =========================

def target_days(cat):
    if cat == "Runner":
        return 5
    elif cat == "Repeater":
        return 3
    else:
        return 1

df["Target_Days"] = df["Category"].apply(target_days)
df["Target_Inventory"] = df["Daily_Demand"] * df["Target_Days"]

# =========================
# PRODUCTION QTY
# =========================

df["Production_Qty"] = np.where(
    df["Trigger"] == 1,
    df["Target_Inventory"] - df["Inventory"],
    0
)

df["Production_Qty"] = df["Production_Qty"].clip(lower=0)

# =========================
# HOURS REQUIRED
# =========================

df["Hours_Required"] = df["Production_Qty"] / df["Rate"]

# =========================
# PRIORITY
# =========================

priority_map = {"Runner":3,"Repeater":2,"Stranger":1}
df["Priority"] = df["Category"].map(priority_map)

df = df.sort_values(by="Priority", ascending=False)

# =========================
# ASSIGN BEST MACHINE
# =========================

df = df.sort_values(by="Hours_Required")

df = df.drop_duplicates(subset=["Part"], keep="first")

# =========================
# MACHINE LOAD CHECK
# =========================

machine_load = df.groupby("Machine")["Hours_Required"].sum().reset_index()
machine_load["Overload"] = machine_load["Hours_Required"] - MACHINE_CAPACITY

# =========================
# HANDLE OVERLOAD
# =========================

for machine in machine_load["Machine"]:

    overload = machine_load.loc[machine_load["Machine"] == machine, "Overload"].values[0]

    if overload > 0:

        subset = df[df["Machine"] == machine]

        strangers = subset[subset["Category"]=="Stranger"]

        for idx in strangers.index:
            if overload <= 0:
                break
            overload -= df.loc[idx,"Hours_Required"]
            df.loc[idx,"Production_Qty"] = 0
            df.loc[idx,"Hours_Required"] = 0

        repeaters = subset[subset["Category"]=="Repeater"]

        for idx in repeaters.index:
            if overload <= 0:
                break
            overload -= df.loc[idx,"Hours_Required"]
            df.loc[idx,"Production_Qty"] = 0
            df.loc[idx,"Hours_Required"] = 0

# =========================
# FINAL OUTPUT
# =========================

df["Produce_Today"] = np.where(df["Production_Qty"]>0,"YES","NO")

df_final = df[[
    "Part",
    "Category",
    "Inventory",
    "Production_Qty",
    "Machine",
    "Hours_Required",
    "Produce_Today"
]]

df_final.to_excel("APS_Plan.xlsx", index=False)

print("APS Plan Generated")

In [ ]:
import pandas as pd
import numpy as np

WORKING_DAYS = 24
MACHINE_CAPACITY = 22

# =========================
# LOAD BOTH SHEETS
# =========================

file_path = "C:/Users/Ex0164/Book1.xlsx"

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")

# =========================
# FUNCTION TO EXPAND MACHINES
# =========================

def expand_machines(df):

    base_cols = ["Part","Inventory","Indent","Category","Cycle time","Cavity"]

    machine_cols = [col for col in df.columns if col.startswith("Machine")]

    df_list = []

    for machine_col in machine_cols:
        temp = df[base_cols].copy()
        temp["Machine"] = df[machine_col]

        temp = temp[temp["Machine"].notna()]
        temp = temp[temp["Machine"] != ""]

        df_list.append(temp)

    return pd.concat(df_list, ignore_index=True)

# Expand both sheets
hz_long = expand_machines(hz)
vt_long = expand_machines(vt)

df = pd.concat([hz_long, vt_long], ignore_index=True)

# =========================
# CALCULATE RATE
# =========================

df["Rate"] = (3600 / df["Cycle time"]) * df["Cavity"]

# =========================
# MRP LOGIC
# =========================

df["Daily_Demand"] = df["Indent"] / WORKING_DAYS
df["Safety"] = df["Daily_Demand"] * 3
df["Gap"] = df["Safety"] - df["Inventory"]
df["Trigger"] = np.where(df["Gap"] > 0, 1, 0)

# =========================
# CATEGORY TARGET
# =========================

def target_days(cat):
    if cat == "Runner":
        return 5
    elif cat == "Repeater":
        return 3
    else:
        return 1

df["Target_Days"] = df["Category"].apply(target_days)
df["Target_Inventory"] = df["Daily_Demand"] * df["Target_Days"]

# =========================
# PRODUCTION QTY
# =========================

df["Production_Qty"] = np.where(
    df["Trigger"] == 1,
    df["Target_Inventory"] - df["Inventory"],
    0
)

df["Production_Qty"] = df["Production_Qty"].clip(lower=0)

# =========================
# HOURS REQUIRED
# =========================

df["Hours_Required"] = df["Production_Qty"] / df["Rate"]

# =========================
# PRIORITY
# =========================

priority_map = {"Runner":3,"Repeater":2,"Stranger":1}
df["Priority"] = df["Category"].map(priority_map)

df = df.sort_values(by=["Priority","Hours_Required"], ascending=[False,True])

# =========================
# SELECT BEST MACHINE
# =========================

df = df.drop_duplicates(subset=["Part"], keep="first")

# =========================
# MACHINE LOAD CHECK
# =========================

machine_load = df.groupby("Machine")["Hours_Required"].sum().reset_index()
machine_load["Overload"] = machine_load["Hours_Required"] - MACHINE_CAPACITY

# =========================
# HANDLE OVERLOAD
# =========================

for machine in machine_load["Machine"]:

    overload = machine_load.loc[machine_load["Machine"] == machine, "Overload"].values[0]

    if overload > 0:

        subset = df[df["Machine"] == machine]

        # Remove Stranger first
        strangers = subset[subset["Category"]=="Stranger"]

        for idx in strangers.index:
            if overload <= 0:
                break
            overload -= df.loc[idx,"Hours_Required"]
            df.loc[idx,"Production_Qty"] = 0
            df.loc[idx,"Hours_Required"] = 0

        # Then Repeater
        repeaters = subset[subset["Category"]=="Repeater"]

        for idx in repeaters.index:
            if overload <= 0:
                break
            overload -= df.loc[idx,"Hours_Required"]
            df.loc[idx,"Production_Qty"] = 0
            df.loc[idx,"Hours_Required"] = 0

# =========================
# FINAL OUTPUT
# =========================

df["Produce_Today"] = np.where(df["Production_Qty"]>0,"YES","NO")

df_final = df[[
    "Part",
    "Category",
    "Inventory",
    "Production_Qty",
    "Machine",
    "Hours_Required",
    "Produce_Today"
]]

df_final.to_excel("APS_Plan.xlsx", index=False)

print("APS Plan Generated Successfully")

In [ ]:
import pandas as pd
import numpy as np

WORKING_DAYS = 24
MACHINE_CAPACITY = 22
MIN_RUN_HOURS = 4

file_path = "C:/Users/Ex0164/Book1.xlsx"

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")

# =========================
# EXPAND MACHINES
# =========================

def expand_machines(df):

    base_cols = ["Part","Inventory","Indent","Category","Cycle time","Cavity"]
    machine_cols = [col for col in df.columns if col.startswith("Machine")]

    df_list = []

    for machine_col in machine_cols:
        temp = df[base_cols].copy()
        temp["Machine"] = df[machine_col]

        temp = temp[temp["Machine"].notna()]
        temp = temp[temp["Machine"] != ""]

        df_list.append(temp)

    return pd.concat(df_list, ignore_index=True)

# =========================
# APS LOGIC FUNCTION
# =========================

def run_aps(df):

    df["Rate"] = (3600 / df["Cycle time"]) * df["Cavity"]

    df["Daily_Demand"] = df["Indent"] / WORKING_DAYS
    df["Safety"] = df["Daily_Demand"] * 3
    df["Gap"] = df["Safety"] - df["Inventory"]

    df["Trigger"] = np.where(df["Gap"] > 0, 1, 0)

    def target_days(cat):
        if cat == "Runner":
            return 5
        elif cat == "Repeater":
            return 3
        else:
            return 1

    df["Target_Days"] = df["Category"].apply(target_days)
    df["Target_Inventory"] = df["Daily_Demand"] * df["Target_Days"]

    df["Production_Qty"] = np.where(
        df["Trigger"] == 1,
        df["Target_Inventory"] - df["Inventory"],
        0
    )

    df["Production_Qty"] = df["Production_Qty"].clip(lower=0)

    df["Hours_Required"] = df["Production_Qty"] / df["Rate"]

    priority_map = {"Runner":3,"Repeater":2,"Stranger":1}
    df["Priority"] = df["Category"].map(priority_map)

    df = df.sort_values(by=["Priority","Hours_Required"], ascending=[False,True])

    # =========================
    # MACHINE-WISE ALLOCATION
    # =========================

    final_plan = []

    for machine, group in df.groupby("Machine"):

        remaining_hours = MACHINE_CAPACITY

        for _, row in group.iterrows():

            if remaining_hours <= 0:
                break

            hours = row["Hours_Required"]

            if hours < MIN_RUN_HOURS:
                hours = MIN_RUN_HOURS

            if hours > remaining_hours:
                continue

            qty = hours * row["Rate"]

            final_plan.append({
                "Part": row["Part"],
                "Category": row["Category"],
                "Machine": machine,
                "Run_Hours": hours,
                "Production_Qty": qty
            })

            remaining_hours -= hours

    final_df = pd.DataFrame(final_plan)

    machine_summary = final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    return final_df, machine_summary

# =========================
# RUN APS FOR BOTH
# =========================

hz_long = expand_machines(hz)
vt_long = expand_machines(vt)

hz_plan, hz_machine = run_aps(hz_long)
vt_plan, vt_machine = run_aps(vt_long)

# =========================
# SAVE OUTPUT
# =========================

with pd.ExcelWriter("APS_Plan.xlsx") as writer:

    hz_plan.to_excel(writer, sheet_name="HZ_Plan", index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan", index=False)

    hz_machine.to_excel(writer, sheet_name="HZ_Machine_Load", index=False)
    vt_machine.to_excel(writer, sheet_name="VT_Machine_Load", index=False)

print("APS Plan Generated with Machine-wise Allocation")

In [ ]:
import pandas as pd
import numpy as np

WORKING_DAYS = 24
MACHINE_CAPACITY = 22
MIN_RUN_HOURS = 4

file_path = "C:/Users/Ex0164/Book1.xlsx"

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")

# =========================
# EXPAND MACHINES
# =========================

def expand_machines(df):

    base_cols = ["Part","Inventory","Indent","Category","Cycle time","Cavity"]
    machine_cols = [col for col in df.columns if col.startswith("Machine")]

    df_list = []

    for machine_col in machine_cols:
        temp = df[base_cols].copy()
        temp["Machine"] = df[machine_col]

        temp = temp[temp["Machine"].notna()]
        temp = temp[temp["Machine"] != ""]

        df_list.append(temp)

    return pd.concat(df_list, ignore_index=True)

# =========================
# SMART APS ENGINE
# =========================

def run_smart_aps(df):

    df["Rate"] = (3600 / df["Cycle time"]) * df["Cavity"]
    df["Daily_Demand"] = df["Indent"] / WORKING_DAYS
    df["Safety"] = df["Daily_Demand"] * 3

    # Current coverage
    df["Coverage"] = df["Inventory"] / df["Daily_Demand"]

    # Risk level
    df["Risk"] = 3 - df["Coverage"]

    df = df[df["Risk"] > 0].copy()

    priority_map = {"Runner":3,"Repeater":2,"Stranger":1}
    df["Priority"] = df["Category"].map(priority_map)

    df = df.sort_values(by=["Priority","Risk"], ascending=[False,False])

    # Build only 1 day per cycle
    df["Build_Qty"] = df["Daily_Demand"]

    df["Hours_Needed"] = df["Build_Qty"] / df["Rate"]

    # =========================
    # PICK BEST MACHINE PER PART
    # =========================

    best_machine = df.loc[df.groupby("Part")["Hours_Needed"].idxmin()]
    df = best_machine.copy()

    # =========================
    # MACHINE-WISE ALLOCATION
    # =========================

    final_plan = []

    for machine, group in df.groupby("Machine"):

        remaining_hours = MACHINE_CAPACITY

        for _, row in group.iterrows():

            if remaining_hours <= 0:
                break

            hours = row["Hours_Needed"]

            if hours < MIN_RUN_HOURS:
                hours = MIN_RUN_HOURS

            if hours > remaining_hours:
                continue

            qty = hours * row["Rate"]

            final_plan.append({
                "Part": row["Part"],
                "Category": row["Category"],
                "Machine": machine,
                "Run_Hours": hours,
                "Production_Qty": qty
            })

            remaining_hours -= hours

    final_df = pd.DataFrame(final_plan)

    machine_summary = final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    return final_df, machine_summary

# =========================
# RUN APS
# =========================

hz_long = expand_machines(hz)
vt_long = expand_machines(vt)

hz_plan, hz_machine = run_smart_aps(hz_long)
vt_plan, vt_machine = run_smart_aps(vt_long)

# =========================
# SAVE OUTPUT
# =========================

with pd.ExcelWriter("APS_Smart_Plan.xlsx") as writer:

    hz_plan.to_excel(writer, sheet_name="HZ_Plan", index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan", index=False)

    hz_machine.to_excel(writer, sheet_name="HZ_Machine_Load", index=False)
    vt_machine.to_excel(writer, sheet_name="VT_Machine_Load", index=False)

print("SMART APS Plan Generated")

In [ ]:
import pandas as pd
import numpy as np

WORKING_DAYS = 24
MACHINE_CAPACITY = 22
MIN_RUN_HOURS = 4

file_path = "C:/Users/Ex0164/Book1.xlsx"

hz = pd.read_excel(file_path, sheet_name="HZ")
vt = pd.read_excel(file_path, sheet_name="VT")

# =========================
# EXPAND MACHINES
# =========================

def expand_machines(df):

    base_cols = ["Part","Inventory","Indent","Category","Cycle time","Cavity"]
    machine_cols = [col for col in df.columns if col.startswith("Machine")]

    df_list = []

    for machine_col in machine_cols:
        temp = df[base_cols].copy()
        temp["Machine"] = df[machine_col]
        df_list.append(temp)

    df_long = pd.concat(df_list, ignore_index=True)

    return df_long

# =========================
# CLEAN DATA
# =========================

def clean_data(df):

    df["Inventory"] = df["Inventory"].fillna(0)
    df["Indent"] = df["Indent"].fillna(0)
    df["Cavity"] = df["Cavity"].fillna(1)
    df["Category"] = df["Category"].fillna("Stranger")

    df = df[df["Machine"].notna()]
    df = df[df["Machine"] != ""]

    df = df[df["Cycle time"].notna()]
    df = df[df["Indent"] > 0]

    return df

# =========================
# SMART APS ENGINE
# =========================

def run_smart_aps(df):

    df["Rate"] = (3600 / df["Cycle time"].replace(0, np.nan)) * df["Cavity"]
    df = df[df["Rate"].notna()]

    df["Daily_Demand"] = df["Indent"] / WORKING_DAYS
    df["Safety"] = df["Daily_Demand"] * 3

    df["Coverage"] = df["Inventory"] / df["Daily_Demand"]
    df["Risk"] = 3 - df["Coverage"]

    df = df[df["Risk"] > 0].copy()

    priority_map = {"Runner":3,"Repeater":2,"Stranger":1}
    df["Priority"] = df["Category"].map(priority_map)

    df = df.sort_values(by=["Priority","Risk"], ascending=[False,False])

    df["Build_Qty"] = df["Daily_Demand"]
    df["Hours_Needed"] = df["Build_Qty"] / df["Rate"]

    # =========================
    # PICK BEST MACHINE PER PART
    # =========================

    best_machine = df.loc[df.groupby("Part")["Hours_Needed"].idxmin()]
    df = best_machine.copy()

    # =========================
    # MACHINE-WISE ALLOCATION
    # =========================

    final_plan = []

    for machine, group in df.groupby("Machine"):

        remaining_hours = MACHINE_CAPACITY

        for _, row in group.iterrows():

            if remaining_hours <= 0:
                break

            hours = row["Hours_Needed"]

            if hours < MIN_RUN_HOURS:
                hours = MIN_RUN_HOURS

            if hours > remaining_hours:
                continue

            qty = hours * row["Rate"]

            final_plan.append({
                "Part": row["Part"],
                "Category": row["Category"],
                "Machine": machine,
                "Run_Hours": round(hours,2),
                "Production_Qty": round(qty,0)
            })

            remaining_hours -= hours

    final_df = pd.DataFrame(final_plan)

    machine_summary = final_df.groupby("Machine")["Run_Hours"].sum().reset_index()

    return final_df, machine_summary

# =========================
# PREPARE DATA
# =========================

hz_long = clean_data(expand_machines(hz))
vt_long = clean_data(expand_machines(vt))

# =========================
# RUN APS
# =========================

hz_plan, hz_machine = run_smart_aps(hz_long)
vt_plan, vt_machine = run_smart_aps(vt_long)

# =========================
# SAVE OUTPUT
# =========================

with pd.ExcelWriter("APS_Smart_Plan.xlsx") as writer:

    hz_plan.to_excel(writer, sheet_name="HZ_Plan", index=False)
    vt_plan.to_excel(writer, sheet_name="VT_Plan", index=False)

    hz_machine.to_excel(writer, sheet_name="HZ_Machine_Load", index=False)
    vt_machine.to_excel(writer, sheet_name="VT_Machine_Load", index=False)

print("SMART APS Plan Generated Successfully")